In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [2]:
CANDIDATE_DB_PATHS = [
    Path("../database/child_health.duckdb"),
    Path("database/child_health.duckdb"),
    Path("/home/harsh_wadhwaniai_org/eda-project/database/child_health.duckdb"),
]

DB_PATH = next((p.resolve() for p in CANDIDATE_DB_PATHS if p.exists()), None)
if DB_PATH is None:
    raise FileNotFoundError(
        "Could not find child_health.duckdb. Run scripts/csv_to_sql.py first to create it."
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected to: {DB_PATH}")

tables = con.sql("SHOW TABLES").df()
print("\nTables in database:")
display(tables)

Connected to: /home/harsh_wadhwaniai_org/eda-project/database/child_health.duckdb

Tables in database:


,name
0,beneficiaries
1,monthly_measurements
2,stunting_lookup
3,underweight_lookup
4,wasting_lookup


In [3]:
row_counts = con.sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
    """
).df()

row_counts["rows"] = row_counts["table_name"].apply(
    lambda t: con.sql(f"SELECT COUNT(*) AS c FROM {t}").df().iloc[0]["c"]
)

display(row_counts)

,table_name,rows
0,beneficiaries,3313714
1,monthly_measurements,9941142
2,stunting_lookup,4444
3,underweight_lookup,4444
4,wasting_lookup,1300


In [4]:
for table_name in row_counts["table_name"]:
    print(f"\n=== {table_name} ===")
    display(con.sql(f"SELECT * FROM {table_name} LIMIT 10").df())


=== beneficiaries ===


,beneficiary_id,dob,gender,birth_height,birth_weight
0,276169768,2023-07-01,M,30.48,3.0
1,272774199,2023-07-10,F,53.00,4.0
2,279315065,2023-07-30,F,49.00,3.1
3,274106612,2023-07-02,M,56.00,3.0
4,275160552,2023-07-29,M,42.30,3.6
5,279093451,2023-07-19,F,52.00,3.2
6,272921167,2023-06-15,M,49.00,2.9
7,274115546,2023-08-09,F,45.00,2.6
8,271316865,2023-07-26,M,49.00,3.0
9,271871062,2023-07-27,M,49.00,2.8



=== monthly_measurements ===


,beneficiary_id,month,status,height,weight,measurement_date,age_days
0,286938541,2024-02-01,active,47.0,3.6,2024-02-14,85
1,273942598,2024-02-01,active,61.0,5.7,2024-02-12,178
2,287769336,2024-02-01,active,55.0,5.0,2024-02-06,95
3,283585440,2024-02-01,active,52.5,4.5,2024-02-03,152
4,287852930,2024-02-01,active,44.3,6.2,2024-02-07,87
5,276470471,2024-02-01,active,60.0,6.6,2024-02-11,163
6,288081455,2024-02-01,active,48.4,4.2,2024-02-02,176
7,283471608,2024-02-01,active,40.8,5.2,2024-02-11,143
8,283811728,2024-02-01,active,54.0,5.1,2024-02-07,148
9,286318246,2024-02-01,active,55.0,4.7,2024-02-05,144



=== stunting_lookup ===


,sex,age_group,day,h_severe_max,h_normal_min
0,F,0-5,0,43.560,45.422
1,F,0-5,1,43.720,45.585
2,F,0-5,2,43.880,45.748
3,F,0-5,3,44.038,45.910
4,F,0-5,4,44.199,46.074
5,F,0-5,5,44.359,46.237
6,F,0-5,6,44.519,46.400
7,F,0-5,7,44.680,46.563
8,F,0-5,8,44.840,46.726
9,F,0-5,9,45.001,46.890



=== underweight_lookup ===


,sex,age_group,day,w_severe_max,w_normal_min
0,F,0-5,0,2.033,2.395
1,F,0-5,1,1.994,2.352
2,F,0-5,2,2.002,2.362
3,F,0-5,3,2.017,2.378
4,F,0-5,4,2.034,2.397
5,F,0-5,5,2.053,2.418
6,F,0-5,6,2.074,2.440
7,F,0-5,7,2.096,2.464
8,F,0-5,8,2.118,2.488
9,F,0-5,9,2.141,2.513



=== wasting_lookup ===


,sex,age_band,height_cm,sam_upper,mam_upper,normal_upper,overweight_upper
0,F,0-2,45.0,1.902,2.066,2.967,3.275
1,F,0-2,45.1,1.915,2.081,2.988,3.298
2,F,0-2,45.2,1.928,2.095,3.008,3.321
3,F,0-2,45.3,1.941,2.109,3.029,3.343
4,F,0-2,45.4,1.954,2.123,3.049,3.366
5,F,0-2,45.5,1.967,2.138,3.070,3.389
6,F,0-2,45.6,1.980,2.152,3.090,3.412
7,F,0-2,45.7,1.993,2.166,3.111,3.434
8,F,0-2,45.8,2.006,2.180,3.131,3.457
9,F,0-2,45.9,2.020,2.195,3.152,3.480
